<a href="https://www.kaggle.com/code/avikdas567/tfidf-nb-ensemble-with-pseudo-labeling-for-bi-rads?scriptVersionId=307621972" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# TF-IDF + NB Ensemble with Pseudo Labeling for BI-RADS NLP

This notebook presents a strong classical NLP solution for the **SPR 2026 Mammography Report Classification** challenge. The goal is to predict **BI-RADS categories (0–6)** from radiology report text using efficient and scalable techniques.

> **OOF Macro F1: ~0.73 (Stratified 5-Fold CV)**

---

## Approach Overview

This solution combines multiple proven NLP techniques:

### 1. Text Preprocessing
- Lowercasing
- Basic normalization (whitespace cleanup)
- No aggressive cleaning to preserve medical terminology

### 2. Feature Engineering

#### Word-level TF-IDF
- `ngram_range=(1,3)`
- Captures semantic patterns and medical phrases

#### Character-level TF-IDF
- `ngram_range=(3,6)`
- Crucial for:
  - Medical abbreviations
  - Misspellings
  - Morphological variations

#### Final Feature Space
- Concatenation of word + char TF-IDF
- Sparse high-dimensional representation

### 3. Naive Bayes Feature Transformation (NB-SVM Trick)

We compute **log-count ratio features** per class:

$$
r = \log \left( \frac{P(x|y=1)}{P(x|y=0)} \right)
$$

- Applied in a **one-vs-rest fashion (7 classes)**
- Enhances linear separability
- Known to significantly boost performance in text tasks

### 4. Model Ensemble

We train multiple models on both:
- Original TF-IDF features
- NB-transformed features

#### Models used:
- Logistic Regression (robust, well-calibrated)
- SGDClassifier (modified huber loss)

#### Final prediction:
- Averaged probabilities across:
  - LR
  - SGD
  - LR (NB features)
  - SGD (NB features)

### 5. Cross Validation

- **Stratified 5-Fold CV**
- Ensures class balance
- Reliable OOF evaluation

### 6. Pseudo Labeling

- Use test predictions as soft labels
- Augment training data
- Retrain final model on:
  - Train + pseudo-labeled test

---

## Why This Works Well

- TF-IDF is highly effective for structured medical text
- Character n-grams capture domain-specific patterns
- NB transformation improves linear models
- Ensemble reduces variance
- Pseudo-labeling leverages test distribution

---

In [1]:
import numpy as np
import pandas as pd
import re
import gc
import time

from scipy.sparse import hstack, vstack

from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import f1_score

from tqdm import tqdm

import warnings
warnings.filterwarnings("ignore")

SEED = 42
N_FOLDS = 5

TRAIN_PATH = "/kaggle/input/competitions/spr-2026-mammography-report-classification/train.csv"
TEST_PATH = "/kaggle/input/competitions/spr-2026-mammography-report-classification/test.csv"

start_total = time.time()

# LOAD DATA

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

id_col = [c for c in test.columns if c.lower() == "id"][0]

# CLEAN TEXT

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"\n", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text

train["report"] = train["report"].apply(clean_text)
test["report"] = test["report"].apply(clean_text)

# TF-IDF

print("Building TF-IDF...")

all_text = pd.concat([train["report"], test["report"]])

word_vec = TfidfVectorizer(
    max_features=150000,
    ngram_range=(1,3),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

char_vec = TfidfVectorizer(
    max_features=120000,
    ngram_range=(3,6),
    analyzer="char_wb",
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_word = word_vec.fit_transform(all_text)
X_char = char_vec.fit_transform(all_text)

X_all = hstack([X_word, X_char]).tocsr()

X_train = X_all[:len(train)]
X_test = X_all[len(train):]

y = train["target"].values

# NB FEATURES

print("Building NB features...")

def get_nb_features(X, y):
    return np.log((X[y==1].sum(0)+1) / (X[y==0].sum(0)+1))

nb_features = []
for c in tqdm(range(7)):
    y_bin = (y == c).astype(int)
    nb_features.append(get_nb_features(X_train, y_bin))

nb_features = np.array(nb_features)

def apply_nb(X):
    return hstack([X.multiply(nb_features[c]) for c in range(7)])

X_train_nb = apply_nb(X_train).tocsr()
X_test_nb = apply_nb(X_test).tocsr()

# TRAIN FUNCTION

def train_models(X, y):
    lr = LogisticRegression(C=3, max_iter=3000, class_weight="balanced", n_jobs=-1)
    sgd = SGDClassifier(loss="modified_huber", alpha=1e-5, max_iter=2000,
                        class_weight="balanced", random_state=SEED)
    lr.fit(X, y)
    sgd.fit(X, y)
    return lr, sgd

# CV LOOP

print("Training...")

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

oof = np.zeros(len(train))
test_preds = np.zeros((len(test), 7))

for fold, (tr_idx, val_idx) in enumerate(tqdm(skf.split(X_train, y), total=N_FOLDS)):
    print(f"\nFold {fold+1}")

    X_tr, X_val = X_train[tr_idx], X_train[val_idx]
    X_tr_nb, X_val_nb = X_train_nb[tr_idx], X_train_nb[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]

    lr, sgd = train_models(X_tr, y_tr)
    lr_nb, sgd_nb = train_models(X_tr_nb, y_tr)

    val_pred = (
        lr.predict_proba(X_val) +
        sgd.predict_proba(X_val) +
        lr_nb.predict_proba(X_val_nb) +
        sgd_nb.predict_proba(X_val_nb)
    ) / 4

    oof[val_idx] = np.argmax(val_pred, axis=1)

    print("F1:", f1_score(y_val, oof[val_idx], average="macro"))

    test_preds += (
        lr.predict_proba(X_test) +
        sgd.predict_proba(X_test) +
        lr_nb.predict_proba(X_test_nb) +
        sgd_nb.predict_proba(X_test_nb)
    ) / 4

print("\nOOF F1:", f1_score(y, oof, average="macro"))

# PSEUDO LABELING

print("Pseudo labeling...")

test_preds /= N_FOLDS
pseudo_labels = np.argmax(test_preds, axis=1)

X_aug = vstack([X_train, X_test])
y_aug = np.concatenate([y, pseudo_labels])

final_model = LogisticRegression(C=3, max_iter=3000, class_weight="balanced", n_jobs=-1)
final_model.fit(X_aug, y_aug)

final_preds = final_model.predict(X_test)

# SUBMISSION

submission = pd.DataFrame({
    id_col: test[id_col],
    "target": final_preds
})

submission.to_csv("submission.csv", index=False)

print("\nDone! Total time:", time.time() - start_total)
display(submission.head())

Building TF-IDF...
Building NB features...


100%|██████████| 7/7 [00:00<00:00, 20.18it/s]


Training...


  0%|          | 0/5 [00:00<?, ?it/s]


Fold 1


...

F1: 0.6999256059321928

Fold 5


100%|██████████| 5/5 [12:10<00:00, 146.17s/it]

F1: 0.7423403279024539

OOF F1: 0.7306166904105043
Pseudo labeling...



Done! Total time: 759.4606552124023


,ID,target
0,Acc0,6
1,Acc2,2
2,Acc4,2
3,Acc10,2
